In [0]:
import pandas as pd

In [0]:
spark.sql("use catalog proyecto_final_prueba")

In [0]:
catalog = spark.sql("select current_catalog()").first()[0]
schema = "gold"
table = "dim_clima"

In [0]:
spark.sql(f"create schema if not exists {catalog}.{schema}")

In [0]:
spark.sql(f"drop table if exists {catalog}.{schema}.{table}")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{schema}.{table} (
    id_clima BIGINT,
    weather_code INT,
    weather_desc STRING
)
""")

In [0]:
df_silver = spark.table(f"{catalog}.silver.weather").toPandas()
df_silver

In [0]:
dim = df_silver[
    ["weather_code", "weather_desc"]
].drop_duplicates().reset_index(drop=True)

dim

In [0]:
dim["id_clima"] = dim.index + 1

In [0]:
columnas_dim = [
    "id_clima",
    "weather_code",
    "weather_desc"
]

dim = dim[columnas_dim]

dim

In [0]:
df_spark = spark.createDataFrame(dim)

df_spark.display()

df_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        f"{catalog}.{schema}.{table}"
    )